## Gold (real-time) vs Polymarket: “What price will gold close at in 2025? ($4000–5000)” 

This notebook:
- Fetches a **live gold price proxy** (COMEX gold futures via Yahoo Finance) and estimates the **year-end close distribution**.
- Pulls **current Polymarket bracket prices** (YES prices) for the event and converts them to **implied probabilities**.
- Compares **model probability** vs **market-implied probability** and computes a simple **expected value (EV)**.

Source market page: `https://polymarket.com/event/what-price-will-gold-close-at-in-2025-4000-5000/will-gold-close-between-4400-and-4500-at-the-end-of-2025-293-543?tid=1767016722291`

**Important**: This is **educational** and not financial advice. Prediction markets have fees/spreads, execution risk, and contract-specific rules. Always read the market rules and size positions assuming you can lose 100%.



In [1]:
import json
import math
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# ---- Configuration ----
POLYMARKET_URL = "https://polymarket.com/event/what-price-will-gold-close-at-in-2025-4000-5000/will-gold-close-between-4400-and-4500-at-the-end-of-2025-293-543?tid=1767016722291"
EVENT_SLUG = "what-price-will-gold-close-at-in-2025-4000-5000"

# Polymarket resolves based on the COMEX Gold Continuous Contract (GC00).
# We'll use Yahoo Finance's continuous/front-month proxy as a practical stand-in.
YAHOO_SYMBOL = "GC=F"  # COMEX Gold Futures

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; research-notebook/1.0)"
})


def _get_json(url: str, *, params: Optional[dict] = None, timeout: int = 20) -> Any:
    r = SESSION.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()


def _get_text(url: str, *, timeout: int = 20) -> str:
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text


@dataclass(frozen=True)
class Bracket:
    label: str
    low: Optional[float]  # inclusive; None means -inf
    high: Optional[float]  # exclusive; None means +inf


def parse_bracket_label(label: str) -> Bracket:
    """Parse labels like '<$4000', '$4400–$4500', '$4800-$4900', '>$5000'."""
    s = label.strip().replace(",", "")
    s = s.replace("–", "-")
    s = re.sub(r"\s+", "", s)

    if s.startswith("<"):
        v = float(re.sub(r"[^0-9.]", "", s))
        return Bracket(label=label, low=None, high=v)

    if s.startswith(">"):
        v = float(re.sub(r"[^0-9.]", "", s))
        return Bracket(label=label, low=v, high=None)

    # range
    m = re.match(r"\$?(\d+(?:\.\d+)?)\-\$?(\d+(?:\.\d+)?)", s)
    if not m:
        raise ValueError(f"Unrecognized bracket label: {label!r}")

    lo = float(m.group(1))
    hi = float(m.group(2))
    return Bracket(label=label, low=lo, high=hi)



In [2]:
def fetch_gold_price_yahoo(symbol: str = YAHOO_SYMBOL) -> Dict[str, Any]:
    """Fetch a live quote from Yahoo Finance's public quote endpoint."""
    url = "https://query1.finance.yahoo.com/v7/finance/quote"
    data = _get_json(url, params={"symbols": symbol})
    result = (data.get("quoteResponse") or {}).get("result") or []
    if not result:
        raise RuntimeError(f"No Yahoo quote data for {symbol!r}: {data}")
    q = result[0]
    return {
        "symbol": q.get("symbol"),
        "price": q.get("regularMarketPrice"),
        "currency": q.get("currency"),
        "timestamp": q.get("regularMarketTime"),
        "exchange": q.get("fullExchangeName") or q.get("exchange"),
        "name": q.get("shortName") or q.get("longName"),
    }


gold_quote = fetch_gold_price_yahoo()
now_utc = datetime.now(timezone.utc)

print(f"Now (UTC): {now_utc.isoformat()}")
print(gold_quote)

S0 = float(gold_quote["price"])  # current price proxy
S0


HTTPError: 401 Client Error: Unauthorized for url: https://query1.finance.yahoo.com/v7/finance/quote?symbols=GC%3DF